# Capturing experiment provenance with Dataerai

[`qick_dataerai`](../qick_lib/qick_dataerai/README.md) automatically records the **provenance** of a QICK experiment into [Dataerai](https://dataerai.com): when a program runs it uploads the **configuration**, the **raw acquired IQ data**, and any **analysis** figures as linked Dataerai *assets*, so the lineage *config → raw data → analysis* travels with the data.

This notebook is **self-contained and runs without a board or a Dataerai login**: it simulates an acquisition and captures it against a small local *recording client* that prints exactly what would be uploaded and linked. The real-hardware version is shown for reference — swap the recording client for a real `dataerai.DataeraiClient` to push to Dataerai for real.

**Provenance edges created** (the edge points from the *derived* asset to its *origin*):

- `raw_data → config`  (relationship type `acquired_with`)
- `analysis → raw_data` (relationship type `analysis_of`)

## 1. Setup & authentication

The capture cells below run **offline** (a local recording client + simulated data), so you can read and run this whole notebook with no account. To push a run to Dataerai *for real* (the **Going live** section at the end), do this one-time setup.

**1. Install** the package and the Dataerai client. `qick[dataerai]` pulls the `qick_dataerai` package and the Dataerai Python SDK (`dataerai-sdk`, imported as `dataerai`); the SDK drives a small local daemon that the **`dataerai-cli`** package bundles:

```bash
pip install qick[dataerai]   # qick_dataerai + the Dataerai Python SDK
pip install dataerai-cli     # the `dataerai` command + the local daemon binary
```

**2. Sign in.** This opens a browser to authenticate and stores credentials in your OS keyring (the login requests read **and write** scope — needed to create projects, upload, and link assets):

```bash
dataerai auth login                              # browser login
# dataerai auth login --device                   # headless: shows a code to enter in a browser
# dataerai auth login --server https://HOST      # target a specific Dataerai environment
```

**3. Verify** your identity, token expiry, and server:

```bash
dataerai auth status
```

After that, `DataeraiClient(binary_path="dataerai")` (use the full path from `which dataerai` if it is not on your `PATH`) connects as you, and the **Going live** cell below works as written.

For this in-repo demo we don't need any of the above — we just add `qick_lib` to the import path so `import qick_dataerai` resolves.

In [1]:
import json
import sys
from pathlib import Path
from types import SimpleNamespace

# Use the in-repo qick_dataerai package (no install needed for this demo).
for cand in [Path.cwd(), *Path.cwd().parents]:
    if (cand / "qick_lib" / "qick_dataerai").is_dir():
        sys.path.insert(0, str(cand / "qick_lib"))
        break

import numpy as np
import matplotlib
matplotlib.use("Agg")          # headless; remove in interactive Jupyter for inline plots
import matplotlib.pyplot as plt

import qick_dataerai
from qick_dataerai import ProvenanceRun, capture_run

print("qick_dataerai", qick_dataerai.__version__, "ready")

qick_dataerai 0.1.0 ready


## 2. The real-hardware flow (reference)

On an actual QICK board you run an experiment and hand its results straight to `capture_run`:

```python
from qick import QickSoc
from dataerai import DataeraiClient
from qick_dataerai import capture_run

soc = QickSoc()
prog = MyT1Program(soc, config)
expt_pts, avg_i, avg_q = prog.acquire(soc, progress=True)
fig = plot_t1(expt_pts, avg_i)

with DataeraiClient(binary_path="/usr/local/bin/dataerai") as client:
    result = capture_run(
        client, prog.cfg, (expt_pts, avg_i, avg_q),
        owner_type="user", owner_id=client.auth_status().user_email,
        soccfg=soc, prog=prog, fig=fig, analysis_mode="non_destructive",
    )
```

The cells below reproduce this **without hardware** so you can run and inspect it now.

## 3. A local recording client

`capture_run` only needs an object with `upload(...)` and `create_relationship(...)` — the two methods the Dataerai SDK exposes. Here is a stand-in that records the calls and prints them; in production you pass a real `DataeraiClient` instead.

In [2]:
class RecordingClient:
    """Stand-in for dataerai.DataeraiClient: records uploads + relationship links."""

    def __init__(self):
        self.uploads, self.links, self._n = [], [], 0

    def __enter__(self):
        return self

    def __exit__(self, *a):
        return False

    def upload(self, local_path, *, title, owner_type, owner_id, collection_id=None,
               description=None, alias=None, tags=None, metadata=None, on_progress=None):
        self._n += 1
        aid = f"asset-{self._n}"
        role = (metadata or {}).get("qick_dataerai_role")
        size = Path(local_path).stat().st_size          # the real temp file written by qick_dataerai
        self.uploads.append({"asset_id": aid, "role": role, "title": title,
                             "suffix": Path(local_path).suffix, "bytes": size, "tags": tags})
        print(f"upload  [{role:9}] {aid}  {Path(local_path).suffix:5} {size:>6} B  '{title}'")
        return SimpleNamespace(asset_id=aid, content_id=f"c-{self._n}",
                               transfer_id=f"t-{self._n}", total_bytes=size, chunk_count=1)

    def create_relationship(self, from_asset_id, to_asset_id, rel_type, *,
                            analysis_mode=None, qualifier_note=None,
                            qualifier_time=None, qualifiers=None):
        self.links.append({"from": from_asset_id, "to": to_asset_id, "type": rel_type,
                           "analysis_mode": analysis_mode})
        mode = f"  [{analysis_mode}]" if analysis_mode else ""
        print(f"link    {from_asset_id} --{rel_type}--> {to_asset_id}{mode}")
        return SimpleNamespace(id=f"rel-{len(self.links)}", type=rel_type, direction="outgoing",
                               from_asset_id=from_asset_id, to_asset_id=to_asset_id,
                               analysis_mode=analysis_mode, related_asset={"id": to_asset_id})

## 4. Simulate a T1 measurement

We synthesize what an `RAveragerProgram.acquire(soc)` call returns for a qubit **T1** (energy-relaxation) sweep: a delay sweep with an exponentially decaying excited-state population. The `(expt_pts, avg_di, avg_dq)` tuple is exactly the shape the QICK averager programs return.

In [3]:
np.random.seed(0)
config = {
    "reps": 2000, "expts": 60,
    "start": 0.0, "step": 0.25,          # delay sweep, in microseconds
    "readout_length": 2.0, "pulse_gain": 12000,
}
delays = config["start"] + np.arange(config["expts"]) * config["step"]   # expt_pts (us)
T1_true = 8.0
pop = np.exp(-delays / T1_true)
avg_i = [pop + 0.03 * np.random.randn(delays.size)]    # one readout channel -> list of arrays
avg_q = [0.03 * np.random.randn(delays.size)]
data = (delays, avg_i, avg_q)            # the (expt_pts, avg_di, avg_dq) acquire() shape

fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot(delays, avg_i[0], "o", ms=4, label="I (data)")
ax.plot(delays, pop, "-", label=f"model: T1 = {T1_true} us")
ax.set_xlabel("delay (us)")
ax.set_ylabel("excited-state population")
ax.set_title("Simulated T1 relaxation")
ax.legend()
fig.tight_layout()
print("acquired", avg_i[0].shape[0], "sweep points on", len(avg_i), "readout channel(s)")

acquired 60 sweep points on 1 readout channel(s)


## 5. Capture the run (one call)

`capture_run` uploads the config, the raw IQ data, and the figure, and wires the provenance edges — all in one call.

In [4]:
client = RecordingClient()
result = capture_run(
    client, config, data,
    owner_type="user", owner_id="demo@example.com",
    fig=fig, analysis_mode="non_destructive",
    tags=["t1-experiment"],
)

upload  [config   ] asset-1  .json    291 B  'QICK run — config'
upload  [raw_data ] asset-2  .npz    1742 B  'QICK run — raw data'
link    asset-2 --acquired_with--> asset-1
upload  [analysis ] asset-3  .png   69226 B  'QICK run — analysis'
link    asset-3 --analysis_of--> asset-2  [non_destructive]


## 6. Inspect the provenance graph

The run created three assets and two typed edges, all sharing one `run_id`.

In [5]:
print("run_id          :", result.run_id)
print("config asset    :", result.config_asset_id)
print("raw data assets :", result.raw_asset_ids)
print("analysis assets :", result.analysis_asset_ids)
print("relationships   :", result.relationship_ids)
print("errors          :", result.errors or "none")

print("\nProvenance edges:")
for ln in client.links:
    print(f"  {ln['from']} --{ln['type']}--> {ln['to']}")

print(f"\nEvery asset is tagged for discovery (qick-run:{result.run_id}):")
for u in client.uploads:
    print(f"  {u['asset_id']} [{u['role']:9}] {u['suffix']:5} tags={u['tags']}")

# Sanity checks (this notebook doubles as a test of the integration).
assert len(client.uploads) == 3
assert [ln["type"] for ln in client.links] == ["acquired_with", "analysis_of"]
assert client.links[0]["to"] == result.config_asset_id
assert client.links[1]["to"] == result.raw_asset_ids[0]
print("\nOK: 3 assets, 2 correctly-directed provenance edges.")

run_id          : f595015fb5c44b6cbd7080c65ca8114d
config asset    : asset-1
raw data assets : ['asset-2']
analysis assets : ['asset-3']
relationships   : ['rel-1', 'rel-2']
errors          : none

Provenance edges:
  asset-2 --acquired_with--> asset-1
  asset-3 --analysis_of--> asset-2

Every asset is tagged for discovery (qick-run:f595015fb5c44b6cbd7080c65ca8114d):
  asset-1 [config   ] .json tags=['t1-experiment', 'qick-run:f595015fb5c44b6cbd7080c65ca8114d', 'qick-dataerai']
  asset-2 [raw_data ] .npz  tags=['t1-experiment', 'qick-run:f595015fb5c44b6cbd7080c65ca8114d', 'qick-dataerai']
  asset-3 [analysis ] .png  tags=['t1-experiment', 'qick-run:f595015fb5c44b6cbd7080c65ca8114d', 'qick-dataerai']

OK: 3 assets, 2 correctly-directed provenance edges.


## 7. Step-by-step capture (more control)

`ProvenanceRun` lets you log each artifact yourself — useful for multiple acquisitions, or attaching an analysis to a specific raw asset.

In [6]:
client2 = RecordingClient()
with ProvenanceRun(client2, owner_type="user", owner_id="demo@example.com",
                   title_prefix="T1 measurement") as run:
    run.log_config(config)
    run.log_acquisition(data=data)
    run.log_analysis(fig, analysis_mode="non_destructive")

print("\nrun_id:", run.result.run_id, "| edges:", len(client2.links))

upload  [config   ] asset-1  .json    291 B  'T1 measurement — config'
upload  [raw_data ] asset-2  .npz    1742 B  'T1 measurement — raw data'
link    asset-2 --acquired_with--> asset-1
upload  [analysis ] asset-3  .png   69226 B  'T1 measurement — analysis'
link    asset-3 --analysis_of--> asset-2  [non_destructive]

run_id: da05c9a1729f43f4ad4a4f0364ed20ea | edges: 2


## 8. What actually gets serialized

The config is written as numpy-aware JSON, the IQ data as a compressed `.npz`, and the figure as a PNG. Here is the real config document the upload would send (truncated):

In [7]:
from qick_dataerai import serialize

with serialize.temp_upload_file(".json") as path:
    serialize.write_config_json(path, config, run_id=result.run_id)
    doc = json.loads(Path(path).read_text())

print(json.dumps(doc, indent=2)[:600], "\n...")

{
  "qick_dataerai": {
    "run_id": "f595015fb5c44b6cbd7080c65ca8114d",
    "role": "config",
    "schema": 1
  },
  "cfg": {
    "reps": 2000,
    "expts": 60,
    "start": 0.0,
    "step": 0.25,
    "readout_length": 2.0,
    "pulse_gain": 12000
  },
  "soccfg": null,
  "program": null
} 
...


## 9. Going live

Once you have installed `dataerai-cli` and run `dataerai auth login` (see **Setup & authentication** above), swap the recording client for a real `DataeraiClient` and pass your actual QICK results. Verify you are signed in with `dataerai auth status` first.

```python
from dataerai import DataeraiClient

# Optionally create a project to hold the run, or use an existing one.
with DataeraiClient(binary_path="dataerai") as client:   # full path if not on PATH
    proj = client.create_project("QICK experiments")
    result = capture_run(
        client, prog.cfg, prog.acquire(soc),
        owner_type="project", owner_id=proj.project_id,
        collection_id=proj.root_collection_id,
        soccfg=soc, prog=prog, fig=fig, analysis_mode="non_destructive",
    )
    print(result.run_id, result.relationship_ids)
```

Then open the project in Dataerai (or search the `qick-run:<run_id>` tag, or `qick-dataerai` for every QICK run) to see the uploaded config / raw data / analysis files, their metadata, and the provenance links in each asset's relationships panel. See the Dataerai *QICK integration* guide and [`qick_dataerai`](../qick_lib/qick_dataerai/README.md) for details.